# Restaurant Success & Opportunity Map

Machine learning project analyzing Yelp restaurant data to predict restaurant success and identify market opportunities.


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix,
                             silhouette_score)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.feature_extraction.text import TfidfVectorizer
import re

print('All libraries loaded successfully.')


## 2. Data Loading


In [ ]:
restaurants = pd.read_csv("restaurants_philadelphia.csv")
reviews = pd.read_csv(r"reviews_philadelphia.csv")
users = pd.read_csv(r"users_philadelphia.csv")
checkins = pd.read_csv(r"checkins_philadelphia.csv")
tips = pd.read_csv(r"tips_philadelphia.csv")
census = pd.read_csv(r"philly_census_data.csv")

print('Restaurants')
display(restaurants.head(3))

print('Reviews')
display(reviews.head(3))

print('Users')
display(users.head(3))


## 3. Cleaning / Preprocessing


In [ ]:
print('Restaurant Missing Values')
print(restaurants.isnull().sum().to_string())
print(f'\nDuplicate rows: {restaurants.duplicated().sum()}')
print(f'Duplicate business_ids: {restaurants["business_id"].duplicated().sum()}')


In [ ]:
restaurants = restaurants.drop_duplicates(subset='business_id')

binary_attr_cols = [c for c in restaurants.columns if restaurants[c].dropna().isin([0,1]).all() and c != 'is_open']
for col in binary_attr_cols:
    restaurants[col] = restaurants[col].fillna(0).astype(int)

if 'price_range' in restaurants.columns:
    restaurants['price_range'] = restaurants['price_range'].fillna(restaurants['price_range'].median())

restaurants['stars']        = pd.to_numeric(restaurants['stars'], errors='coerce')
restaurants['review_count'] = pd.to_numeric(restaurants['review_count'], errors='coerce')
restaurants['is_open']      = pd.to_numeric(restaurants['is_open'], errors='coerce').fillna(1).astype(int)

print(f'Restaurants after cleaning: {restaurants.shape}')
restaurants.dtypes


In [ ]:
if 'categories' in restaurants.columns:
    all_cats = restaurants['categories'].dropna().str.split(',').explode().str.strip()
    top_cats = all_cats.value_counts().head(20)
    print('Top 20 categories:')
    print(top_cats.to_string())

    skip = {'Restaurants', 'Food'}
    useful_cats = [c for c in top_cats.index if c not in skip][:15]
    for cat in useful_cats:
        col_name = 'cat_' + re.sub(r'\W+', '_', cat.lower()).strip('_')
        restaurants[col_name] = restaurants['categories'].fillna('').str.contains(re.escape(cat), case=False).astype(int)
    print(f'\nAdded {len(useful_cats)} category indicator columns.')


## 4. Feature Engineering


In [ ]:
reviews['text_len'] = reviews['text'].fillna('').str.len()

review_agg = reviews.groupby('business_id').agg(
    avg_review_len  = ('text_len', 'mean'),
    num_reviews     = ('review_id', 'count'),
    avg_review_stars= ('stars', 'mean'),
    total_useful    = ('useful', 'sum'),
    total_funny     = ('funny', 'sum'),
    total_cool      = ('cool', 'sum'),
).reset_index()

print(f'Review aggregates: {review_agg.shape}')
review_agg.head()


In [ ]:
df = restaurants.copy()
df['postal_code'] = df['postal_code'].astype(str).str.split('.').str[0]
df = df.merge(review_agg, on='business_id', how='left')

if census is not None and 'postal_code' in census.columns:
    census['postal_code'] = census['postal_code'].astype(str).str.split('.').str[0]
    census_cols = ['population', 'median_income', 'poverty_count', 'bachelor_degree', 'median_age']
    for col in census_cols:
        if col in census.columns:
            census[col] = pd.to_numeric(census[col], errors='coerce')
    df = df.merge(census, on='postal_code', how='left')

for col in review_agg.columns:
    if col != 'business_id' and col in df.columns:
        df[col] = df[col].fillna(0)

print(f'Modeling dataframe: {df.shape}')
df.head()


In [ ]:
if 'wifi' in df.columns:
    wifi_map = {'free': 2, "u'free'": 2, 'paid': 1, "u'paid'": 1, 'no': 0, "u'no'": 0, 'none': 0}
    df['wifi_encoded'] = df['wifi'].map(wifi_map).fillna(0).astype(int)

if 'alcohol' in df.columns:
    alc_map = {'full_bar': 2, "u'full_bar'": 2, 'beer_and_wine': 1, "u'beer_and_wine'": 1,
               'none': 0, "u'none'": 0}
    df['alcohol_encoded'] = df['alcohol'].map(alc_map).fillna(0).astype(int)

if 'noise_level' in df.columns:
    noise_map = {'quiet': 1, "u'quiet'": 1, 'average': 2, "u'average'": 2,
                 'loud': 3, "u'loud'": 3, 'very_loud': 4, "u'very_loud'": 4}
    df['noise_encoded'] = df['noise_level'].map(noise_map).fillna(2).astype(int)

print('Encoded wifi, alcohol, noise level categorical attributes.')


In [ ]:
plt.figure()
df[df['num_reviews'] < 1000].boxplot(column='num_reviews', by='is_open')
plt.title("Review Count vs Restaurant Success (Filtered)")
plt.suptitle("")
plt.xlabel("Success (0 = Closed, 1 = Open)")
plt.ylabel("Number of Reviews")
plt.show()


## 5. NLP Analysis


### 5.1 Clean review text


In [ ]:
sample_size = 50000
reviews_sample = reviews.sample(n=min(sample_size, len(reviews)), random_state=42).copy()
reviews_sample['text_clean'] = reviews_sample['text'].fillna('').str.lower()
reviews_sample['text_clean'] = reviews_sample['text_clean'].apply(
    lambda x: re.sub(r'[^a-z\s]', '', x))

print(f'Sampled {len(reviews_sample)} reviews for NLP analysis.')


### 5.2 Sentiment analysis


In [ ]:
reviews_sample['text_len'] = reviews_sample['text'].fillna('').str.len()
fig, ax = plt.subplots(figsize=(10, 6))

avg_lengths = reviews_sample.groupby('stars')['text_len'].mean()

bars = avg_lengths.plot(
    kind='bar',
    ax=ax,
    color='skyblue',
    edgecolor='white'
)

for i, v in enumerate(avg_lengths):
    ax.text(i, v, f'{v:.0f}', ha='center', va='bottom')

ax.set_title('Average Review Length by Star Rating', fontsize=14)
ax.set_xlabel('Stars')
ax.set_ylabel('Avg Review Length (chars)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()


### 5.3 Topic extraction


In [ ]:
stop_words = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with','by',
    'from','is','it','was','were','are','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might','shall',
    'i','me','my','we','our','you','your','he','she','they','them','their',
    'this','that','these','those','not','no','so','very','just','also','its',
    'if','as','than','can','get','got','go','went','like','one','really',
    'dont','didnt','ive','im','thats','theyre','weve','youre','about','up',
])

all_words = ' '.join(reviews_sample['text_clean']).split()
filtered = [w for w in all_words if w not in stop_words and len(w) > 2]
word_counts = Counter(filtered).most_common(30)

fig, ax = plt.subplots(figsize=(12, 7))
words, counts = zip(*word_counts)
ax.barh(words[::-1], counts[::-1], color=sns.color_palette('magma', 30))
ax.set_title('Top 30 Words in Philadelphia Restaurant Reviews', fontsize=14)
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
reviews_sample['star_bucket'] = reviews_sample['stars'].apply(
    lambda x: 'Low (1-2)' if x <= 2 else ('Mid (3)' if x == 3 else 'High (4-5)'))

nlp_term_rows = []
for bucket in ['Low (1-2)', 'Mid (3)', 'High (4-5)']:
    subset = reviews_sample[reviews_sample['star_bucket'] == bucket]['text_clean']
    if subset.empty:
        continue
    tfidf = TfidfVectorizer(max_features=15, stop_words='english', ngram_range=(1,2))
    tfidf_matrix = tfidf.fit_transform(subset)
    top_terms = tfidf.get_feature_names_out()
    scores = tfidf_matrix.mean(axis=0).A1
    term_scores = sorted(zip(top_terms, scores), key=lambda x: -x[1])
    print(f'\n--- {bucket} Star Reviews: Top TF-IDF Terms ---')
    for term, score in term_scores:
        print(f'  {term:25s} {score:.4f}')
        nlp_term_rows.append({'star_bucket': bucket, 'term': term, 'score': score})

nlp_terms_df = pd.DataFrame(nlp_term_rows)


### 5.4 Save NLP outputs


In [ ]:
word_count_df = pd.DataFrame(word_counts, columns=['term', 'count'])

nlp_outputs = {
    'word_counts': word_count_df,
    'tfidf_terms': nlp_terms_df,
}

display(word_count_df.head(10))
display(nlp_terms_df.head(10))


## 6. Neighborhood Analysis


In [ ]:
neighborhood_summary = (
    df.groupby('postal_code')
      .agg(
          restaurant_count=('business_id', 'nunique'),
          avg_stars=('stars', 'mean'),
          open_rate=('is_open', 'mean'),
          avg_review_count=('review_count', 'mean'),
          median_price=('price_range', 'median'),
          median_income=('median_income', 'median'),
          median_age=('median_age', 'median'),
          population=('population', 'median'),
          poverty_count=('poverty_count', 'median'),
          bachelor_degree=('bachelor_degree', 'median'),
      )
      .reset_index()
)

if 'population' in neighborhood_summary.columns:
    neighborhood_summary['poverty_rate'] = neighborhood_summary['poverty_count'] / neighborhood_summary['population']
    neighborhood_summary['bachelor_rate'] = neighborhood_summary['bachelor_degree'] / neighborhood_summary['population']

neighborhood_summary = neighborhood_summary.sort_values(
    ['restaurant_count', 'avg_stars'], ascending=[False, False]
)

display(neighborhood_summary.head(15))


## 7. Restaurant Success Prediction


### 7.1 Define success


In [ ]:
success_target = 'is_open'
print('Success target: is_open')
print('1 = open, 0 = closed')


### 7.2 Select features


In [ ]:
potential_features = [
    'review_count', 'price_range', 'latitude', 'longitude',
    'restaurants_delivery', 'restaurants_takeout', 'restaurants_reservations',
    'outdoor_seating', 'good_for_kids', 'bike_parking',
    'wifi_encoded', 'alcohol_encoded', 'noise_encoded',
    'avg_review_len', 'num_reviews', 'avg_review_stars',
    'total_useful', 'total_funny', 'total_cool',
]

cat_cols = [c for c in df.columns if c.startswith('cat_')]
potential_features.extend(cat_cols)

feature_cols = [c for c in potential_features if c in df.columns]
print(f'Using {len(feature_cols)} features: {feature_cols}')

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


### 7.3 Train test split


In [ ]:
clf_features = [c for c in feature_cols if c != 'avg_review_stars']

X_clf = df[clf_features].copy()
y_clf = df['is_open'].copy()

mask = y_clf.notna()
X_clf, y_clf = X_clf[mask], y_clf[mask].astype(int)

print(f'Class distribution:\n{y_clf.value_counts().to_string()}')
print(f'Closed rate: {(y_clf==0).mean():.1%}')

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c)
X_test_c_sc  = scaler_c.transform(X_test_c)

print(f'\nTraining: {X_train_c.shape}  |  Test: {X_test_c.shape}')


### 7.4 Logistic Regression


In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_c_sc, y_train_c)
y_pred_log = log_reg.predict(X_test_c_sc)


### 7.5 Decision Tree / Random Forest


In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10,
                                 class_weight='balanced', random_state=42, n_jobs=-1)
rf_clf.fit(X_train_c, y_train_c)
y_pred_rf_c = rf_clf.predict(X_test_c)


In [ ]:
imp_clf = pd.Series(rf_clf.feature_importances_, index=clf_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(imp_clf)*0.3)))
imp_clf.tail(20).plot(kind='barh', ax=ax, color='red')
ax.set_title('Top 20 Feature Importances — Restaurant Survival (Random Forest)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()


### 7.6 Compare results


In [ ]:
def clf_metrics(name, y_true, y_pred):
    return {
        'Model': name,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
    }

results_clf = pd.DataFrame([
    clf_metrics('Logistic Regression', y_test_c, y_pred_log),
    clf_metrics('Random Forest',       y_test_c, y_pred_rf_c),
])
display(results_clf)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, y_pred) in zip(axes, [('Logistic Regression', y_pred_log), ('Random Forest', y_pred_rf_c)]):
    cm = confusion_matrix(y_test_c, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Closed','Open'], yticklabels=['Closed','Open'])
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()


### 7.7 Save success predictions


In [ ]:
success_predictions = X_test_c.copy()
success_predictions['actual_is_open'] = y_test_c.values
success_predictions['log_reg_pred'] = y_pred_log
success_predictions['rf_pred'] = y_pred_rf_c
success_predictions['log_reg_prob_open'] = log_reg.predict_proba(X_test_c_sc)[:, 1]
success_predictions['rf_prob_open'] = rf_clf.predict_proba(X_test_c)[:, 1]

display(success_predictions.head(10))


## 8. Opportunity Detection


### 8.1 Build neighborhood dataset


In [ ]:
opportunity_base = neighborhood_summary.copy()
display(opportunity_base.head(10))


### 8.2 Clustering


In [ ]:
cluster_features = [c for c in feature_cols if c in df.columns]
X_cluster = df[cluster_features].fillna(0).copy()

scaler_k = StandardScaler()
X_cluster_sc = scaler_k.fit_transform(X_cluster)

inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_sc, km.labels_, sample_size=min(5000, len(X_cluster_sc))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, sil_scores, 'rs-')
axes[1].set_title('Silhouette Score by k', fontsize=13)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f'Best k by silhouette: {best_k}')


In [ ]:
chosen_k = max(best_k, 3)  # use at least 3 for interpretability
km_final = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_cluster_sc)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster_sc)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'], cmap='viridis',
                     alpha=0.6, s=20, edgecolors='white', linewidth=0.3)
ax.set_title(f'Restaurant Clusters (k={chosen_k}) — PCA Projection', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()


In [ ]:
profile_cols = ['stars', 'review_count', 'price_range', 'is_open',
                'avg_review_len', 'num_reviews']
profile_cols = [c for c in profile_cols if c in df.columns]

cluster_profile = df.groupby('cluster')[profile_cols].mean().round(2)
cluster_profile['count'] = df.groupby('cluster').size().values
display(cluster_profile)


### 8.3 Demand supply scoring


In [ ]:
opportunity_df = neighborhood_summary.copy()

for col in ['avg_stars', 'open_rate', 'avg_review_count', 'median_income', 'bachelor_rate']:
    if col in opportunity_df.columns:
        opportunity_df[col] = pd.to_numeric(opportunity_df[col], errors='coerce')

opportunity_df['restaurant_density_rank'] = 1 - opportunity_df['restaurant_count'].rank(pct=True)
opportunity_df['quality_rank'] = opportunity_df['avg_stars'].rank(pct=True)
opportunity_df['survival_rank'] = opportunity_df['open_rate'].rank(pct=True)
opportunity_df['demand_rank'] = opportunity_df['avg_review_count'].rank(pct=True)
opportunity_df['income_rank'] = opportunity_df['median_income'].fillna(opportunity_df['median_income'].median()).rank(pct=True)
opportunity_df['education_rank'] = opportunity_df['bachelor_rate'].fillna(opportunity_df['bachelor_rate'].median()).rank(pct=True)

opportunity_df['opportunity_score'] = (
    0.20 * opportunity_df['restaurant_density_rank'] +
    0.25 * opportunity_df['quality_rank'] +
    0.20 * opportunity_df['survival_rank'] +
    0.15 * opportunity_df['demand_rank'] +
    0.10 * opportunity_df['income_rank'] +
    0.10 * opportunity_df['education_rank']
)

opportunity_df = opportunity_df[
    (opportunity_df['restaurant_count'] >= 3) &
    (opportunity_df['avg_stars'] >= df['stars'].median()) &
    (opportunity_df['open_rate'] >= df['is_open'].mean())
].sort_values('opportunity_score', ascending=False)

opportunity_table = opportunity_df[
    ['postal_code', 'restaurant_count', 'avg_stars', 'open_rate', 'avg_review_count', 'median_income', 'opportunity_score']
].head(10)

display(opportunity_table)


### 8.4 Find cuisine gaps


In [ ]:
raw_restaurants = pd.read_csv('restaurants_philadelphia.csv')

if not opportunity_table.empty and 'categories' in raw_restaurants.columns and 'postal_code' in raw_restaurants.columns:
    target_zips = (
        opportunity_table['postal_code']
        .fillna(0)
        .astype(float)
        .astype(int)
        .astype(str)
        .tolist()
    )
    
    cuisine_gap_source = raw_restaurants[
        raw_restaurants['postal_code']
        .fillna(0)
        .astype(float)
        .astype(int)
        .astype(str)
        .isin(target_zips)
    ].copy()
    
    cuisine_counts = (
        cuisine_gap_source['categories']
        .dropna()
        .str.split(',')
        .explode()
        .str.strip()
    )
    
    cuisine_counts = cuisine_counts[~cuisine_counts.isin(['Restaurants', 'Food', ''])]
    
    cuisine_gap_table = cuisine_counts.value_counts().reset_index()
    cuisine_gap_table.columns = ['category', 'restaurant_count']
    cuisine_gap_table = cuisine_gap_table.sort_values(
        ['restaurant_count', 'category'],
        ascending=[False, True]
    ).head(15)

else:
    cuisine_gap_table = pd.DataFrame(columns=['category', 'restaurant_count'])

display(cuisine_gap_table)


### 8.5 Save opportunity results


In [ ]:
opportunity_results = {
    'neighborhood_summary': neighborhood_summary,
    'cluster_profile': cluster_profile,
    'opportunity_table': opportunity_table,
    'cuisine_gap_table': cuisine_gap_table,
}

display(opportunity_table)


## 9. Final Output Tables


In [ ]:
final_outputs = {
    'nlp_word_counts': word_count_df.head(15),
    'nlp_tfidf_terms': nlp_terms_df.head(15),
    'neighborhood_summary': neighborhood_summary.head(10),
    'classification_results': results_clf.sort_values('F1', ascending=False),
    'success_predictions': success_predictions.head(10),
    'cluster_profile': cluster_profile,
    'opportunity_table': opportunity_table,
    'cuisine_gap_table': cuisine_gap_table,
}

for name, table in final_outputs.items():
    print(f'\n {name}')
    display(table)


In [ ]:
plt.figure()

df_filtered = df[df['num_reviews'] < 1000]

plt.scatter(df_filtered['num_reviews'], df_filtered['avg_review_stars'], alpha=0.4)

plt.xlabel("Customer Demand (Number of Reviews)")
plt.ylabel("Customer Satisfaction (Average Rating)")
plt.title("Restaurant Demand vs Customer Satisfaction")

plt.show()
